# 0. Возобновляемая мультимодальная векторизация постов компаний

Автономный notebook формирует неизменяемый snapshot постов, вычисляет эмбеддинги
независимыми GPU-батчами и подтверждает их в PostgreSQL транзакциями по 500 строк.
Источник истины для resume — только `post_embeddings`; локальные журналы служат аудиту.

Схема: `секреты → read-only preflight → immutable snapshot → DB resume → encoder →
RAM buffer → transaction/COMMIT → аудит → quality report/summary`.

По умолчанию включён безопасный `smoke`: синтетические данные, Mock-энкодер и fake DB.
Production-запись разрешается только при одновременных значениях
`RUN_MODE="production"`, `APPLY_DB_WRITES=True` и
`CONFIRMATION="WRITE_POST_EMBEDDINGS"`. Notebook не выполняет DDL, миграции, сбор VK,
LLM-вызовы или кластеризацию. Секреты не попадают в артефакты и логи.


## 1. Установка и проверка зависимостей


In [ ]:
import importlib.util
import subprocess
import sys

REQUIRED_PACKAGES = {
    "pydantic": "pydantic>=2,<3",
    "pydantic_settings": "pydantic-settings>=2,<3",
    "sqlalchemy": "SQLAlchemy>=2,<3",
    "asyncpg": "asyncpg>=0.29",
    "numpy": "numpy>=1.26",
    "PIL": "Pillow>=10",
    "psutil": "psutil>=5.9",
    "dotenv": "python-dotenv>=1",
    "sshtunnel": "sshtunnel>=0.4",
    "paramiko": "paramiko>=3",
}

missing = [spec for module, spec in REQUIRED_PACKAGES.items() if importlib.util.find_spec(module) is None]
if missing:
    print("Устанавливаются только отсутствующие зависимости:", ", ".join(missing))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
else:
    print("Обязательные зависимости уже установлены; стек PyTorch не изменяется.")


## 2. Пользовательская конфигурация запуска


In [ ]:
from __future__ import annotations

GPU_BATCH_SIZE = 32
GPU_MIN_BATCH_SIZE = 8
DB_BATCH_SIZE = 500
EMBEDDING_DIM = 2048
RESOURCE_SAMPLE_INTERVAL_SECONDS = 10
RESUME = True
RUN_MODE = "smoke"
APPLY_DB_WRITES = False
CONFIRMATION = ""
FORCE_NEW_RUN = False

DATASET_NAME = "approved_company_posts_180d_100_per_group"
MODEL_NAME = "mock-multimodal-encoder" if RUN_MODE == "smoke" else "Qwen/Qwen3-VL-Embedding-2B"
MODEL_REVISION = "main"
WINDOW_DAYS = 180
MAX_POSTS_PER_GROUP = 100
SNAPSHOT_SQL_BATCH_SIZE = 500
ATTACHMENT_POST_IDS_BATCH_SIZE = 250
QUALITY_RESERVOIR_SIZE = 10_000
EXPORT_EMBEDDING_SHARDS = False
DB_MAX_RETRIES = 4
EXECUTE_PIPELINE = False  # Измените на True только после проверки конфигурации.
RANDOM_SEED = 42

def db_writes_authorized() -> bool:
    """Проверить точное тройное подтверждение реальных записей в БД."""
    return (
        RUN_MODE == "production"
        and APPLY_DB_WRITES is True
        and CONFIRMATION == "WRITE_POST_EMBEDDINGS"
    )

if APPLY_DB_WRITES and not db_writes_authorized():
    raise RuntimeError("Запись запрещена: требуется production + APPLY_DB_WRITES=True + точное confirmation.")


## 3. Определение среды: Colab, local, supercomputer


In [ ]:
import os
import platform
import socket
from pathlib import Path

def detect_environment() -> str:
    """Определить поддерживаемую среду выполнения."""
    if "google.colab" in sys.modules:
        return "colab"
    if Path("/persistent_volume").exists() or os.getenv("VK_VECTORIZATION_ENV") == "supercomputer":
        return "supercomputer"
    return "local"

def find_project_root(start: Path | None = None) -> Path:
    """Найти корень проекта по маркерам, не полагаясь только на cwd."""
    candidates = [start or Path.cwd(), Path.cwd(), Path(__file__).parent if "__file__" in globals() else Path.cwd()]
    for candidate in candidates:
        for path in [candidate.resolve(), *candidate.resolve().parents]:
            if (path / "pyproject.toml").exists() and (path / "config" / "keywords.yml").exists():
                return path
    raise RuntimeError("Не удалось надёжно определить корень локального проекта.")

ENVIRONMENT = detect_environment()
print(f"Среда: {ENVIRONMENT}; узел: {socket.gethostname()}; ОС: {platform.platform()}")


## 4. Настройка постоянного хранилища


In [ ]:
import tempfile

def verify_writable_storage(path: Path) -> None:
    """Проверить запись и чтение небольшого файла в целевом хранилище."""
    path.mkdir(parents=True, exist_ok=True)
    probe = path / ".storage_probe"
    payload = "storage-ok"
    try:
        probe.write_text(payload, encoding="utf-8")
        if probe.read_text(encoding="utf-8") != payload:
            raise OSError("контрольное содержимое не совпало")
    finally:
        probe.unlink(missing_ok=True)

if ENVIRONMENT == "colab":
    STORAGE_ROOT = Path("/content/drive/MyDrive/vk-research-collector/vectorization_runs")
elif ENVIRONMENT == "supercomputer":
    persistent_root = Path("/persistent_volume")
    if not persistent_root.exists():
        raise RuntimeError("На суперкомпьютере отсутствует /persistent_volume; запуск остановлен.")
    STORAGE_ROOT = persistent_root / "vk-research-collector/vectorization_runs"
    cache_root = persistent_root / "vk-research-collector/cache"
    temp_root = persistent_root / "vk-research-collector/tmp"
    for name, value in {
        "HF_HOME": cache_root / "huggingface",
        "TORCH_HOME": cache_root / "torch",
        "XDG_CACHE_HOME": cache_root / "xdg",
        "TMPDIR": temp_root,
        "TEMP": temp_root,
        "TMP": temp_root,
    }.items():
        value.mkdir(parents=True, exist_ok=True)
        os.environ[name] = str(value)
    tempfile.tempdir = str(temp_root)
else:
    STORAGE_ROOT = find_project_root() / "exports" / "vectorization_runs"

verify_writable_storage(STORAGE_ROOT)
print(f"Постоянное хранилище: {STORAGE_ROOT}")


In [ ]:
# Отдельная идемпотентная ячейка подключения Google Drive.
if ENVIRONMENT == "colab":
    from google.colab import drive  # type: ignore[import-not-found]
    if not Path("/content/drive/MyDrive").exists():
        drive.mount("/content/drive")
    verify_writable_storage(STORAGE_ROOT)
    print("Google Drive подключён и проверен.")
else:
    print("Подключение Google Drive не требуется.")


## 5. Безопасная загрузка секретов


In [ ]:
import re
from urllib.parse import quote, urlsplit, urlunsplit

from dotenv import load_dotenv

load_dotenv(override=False)

SECRET_NAMES = (
    "DATABASE_URL", "DB_HOST", "DB_PORT", "DB_NAME", "DB_USER", "DB_PASSWORD",
    "DB_PASSWORD_FILE", "DB_SSL_MODE", "DB_SSL_CA_FILE", "SSH_ENABLED", "SSH_HOST",
    "SSH_PORT", "SSH_USER", "SSH_PRIVATE_KEY", "SSH_PRIVATE_KEY_FILE",
)

def read_secret(name: str, *, required: bool = False) -> str | None:
    """Прочитать секрет из Colab Secrets, env/.env или файла `<NAME>_FILE`."""
    value: str | None = None
    if ENVIRONMENT == "colab":
        try:
            from google.colab import userdata  # type: ignore[import-not-found]
            value = userdata.get(name)
        except Exception:
            value = None
    value = value or os.getenv(name)
    secret_file = os.getenv(f"{name}_FILE")
    if not value and secret_file:
        path = Path(secret_file).expanduser()
        if not path.is_file():
            raise RuntimeError(f"Файл секрета для {name} не найден.")
        value = path.read_text(encoding="utf-8").strip()
    if required and not value:
        raise RuntimeError(f"Не задан обязательный секрет {name}.")
    return value

def mask_database_url(value: str | None) -> str:
    """Вернуть безопасное описание URL без пароля."""
    if not value:
        return "<не задан>"
    try:
        parts = urlsplit(value)
        host = parts.hostname or "<host>"
        port = f":{parts.port}" if parts.port else ""
        user = parts.username or "<user>"
        return urlunsplit((parts.scheme, f"{user}:***@{host}{port}", parts.path, "", ""))
    except Exception:
        return "<DATABASE_URL скрыт>"

class SecretRedactor:
    """Редактировать известные секреты и URL в строках."""
    def __init__(self, values: list[str] | None = None) -> None:
        self.values = sorted({v for v in (values or []) if v}, key=len, reverse=True)

    def redact(self, message: str) -> str:
        result = str(message)
        for value in self.values:
            result = result.replace(value, "***")
        result = re.sub(r"(?i)postgres(?:ql)?(?:\+asyncpg)?://[^\s]+", "<DATABASE_URL скрыт>", result)
        result = re.sub(r"-----BEGIN[\s\S]*?-----END[^-]*PRIVATE KEY-----", "<SSH KEY скрыт>", result)
        return result

SECRETS = {name: read_secret(name) for name in SECRET_NAMES}
REDACTOR = SecretRedactor([value for value in SECRETS.values() if value])
print("Секреты загружены из разрешённых источников; значения не выводятся.")


## 6. Логирование и монитор ресурсов


In [ ]:
import csv
import json
import logging
import shutil
import subprocess
import threading
import time
from collections import defaultdict
from collections.abc import Iterator, Sequence
from datetime import UTC, datetime
from logging.handlers import RotatingFileHandler
from typing import Any

import psutil

def utc_now() -> datetime:
    """Вернуть текущее время UTC."""
    return datetime.now(UTC)

def append_jsonl(path: Path, payload: dict[str, Any]) -> None:
    """Добавить одну JSON-запись с flush/fsync."""
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("a", encoding="utf-8") as stream:
        stream.write(json.dumps(payload, ensure_ascii=False, default=str) + "\n")
        stream.flush()
        with contextlib.suppress(OSError):
            os.fsync(stream.fileno())

class SecretRedactionFilter(logging.Filter):
    """Не допускать известные секреты в logging records."""
    def filter(self, record: logging.LogRecord) -> bool:
        record.msg = REDACTOR.redact(record.getMessage())
        record.args = ()
        return True

def configure_logger(run_log: Path) -> logging.Logger:
    """Настроить русский человекочитаемый журнал с ротацией."""
    logger = logging.getLogger("vectorization")
    logger.setLevel(logging.INFO)
    logger.handlers.clear()
    handler = RotatingFileHandler(run_log, maxBytes=5_000_000, backupCount=3, encoding="utf-8")
    handler.setFormatter(logging.Formatter("%(asctime)sZ %(levelname)s %(message)s"))
    handler.addFilter(SecretRedactionFilter())
    logger.addHandler(handler)
    return logger

def collect_gpu_metrics() -> dict[str, Any]:
    """Безопасно собрать GPU-метрики без сети и энергопотребления."""
    metrics: dict[str, Any] = {
        "gpu_utilization_percent": None, "gpu_temperature_c": None,
        "vram_total_bytes": None, "vram_free_bytes": None,
        "vram_allocated_bytes": None, "vram_reserved_bytes": None,
        "vram_max_allocated_bytes": None,
    }
    try:
        import torch
        if torch.cuda.is_available():
            free, total = torch.cuda.mem_get_info()
            metrics.update(
                vram_total_bytes=total, vram_free_bytes=free,
                vram_allocated_bytes=torch.cuda.memory_allocated(),
                vram_reserved_bytes=torch.cuda.memory_reserved(),
                vram_max_allocated_bytes=torch.cuda.max_memory_allocated(),
            )
    except Exception:
        pass
    if shutil.which("nvidia-smi"):
        try:
            output = subprocess.check_output(
                ["nvidia-smi", "--query-gpu=utilization.gpu,temperature.gpu", "--format=csv,noheader,nounits"],
                text=True, timeout=3,
            ).splitlines()[0]
            utilization, temperature = [part.strip() for part in output.split(",")[:2]]
            metrics["gpu_utilization_percent"] = float(utilization)
            metrics["gpu_temperature_c"] = float(temperature)
        except Exception:
            pass
    return metrics

class ResourceMonitor:
    """Периодически сохранять CPU/RAM/disk/GPU и счётчики runner."""
    def __init__(self, destination: Path, storage_path: Path, state: dict[str, Any], interval: int = 10) -> None:
        self.destination, self.storage_path, self.state, self.interval = destination, storage_path, state, interval
        self.stop_event = threading.Event()
        self.thread: threading.Thread | None = None

    def start(self) -> None:
        if self.thread and self.thread.is_alive():
            return
        self.thread = threading.Thread(target=self._run, name="resource-monitor", daemon=True)
        self.thread.start()

    def stop(self) -> None:
        self.stop_event.set()
        if self.thread:
            self.thread.join(timeout=max(2, self.interval + 1))

    def sample(self) -> dict[str, Any]:
        process, memory, swap = psutil.Process(), psutil.virtual_memory(), psutil.swap_memory()
        disk = shutil.disk_usage(self.storage_path)
        row = {
            "timestamp_utc": utc_now().isoformat(), "process_cpu_percent": process.cpu_percent(),
            "system_cpu_percent": psutil.cpu_percent(), "rss_bytes": process.memory_info().rss,
            "vms_bytes": process.memory_info().vms, "ram_total_bytes": memory.total,
            "ram_available_bytes": memory.available, "swap_total_bytes": swap.total,
            "swap_used_bytes": swap.used, "disk_total_bytes": disk.total,
            "disk_used_bytes": disk.used, "disk_free_bytes": disk.free, **collect_gpu_metrics(), **self.state,
        }
        self.state["peak_ram_bytes"] = max(int(self.state.get("peak_ram_bytes", 0)), int(row["rss_bytes"]))
        self.state["peak_vram_bytes"] = max(int(self.state.get("peak_vram_bytes", 0)), int(row.get("vram_max_allocated_bytes") or 0))
        return row

    def _run(self) -> None:
        while not self.stop_event.is_set():
            row = self.sample()
            self.destination.parent.mkdir(parents=True, exist_ok=True)
            exists = self.destination.exists() and self.destination.stat().st_size > 0
            with self.destination.open("a", newline="", encoding="utf-8") as stream:
                writer = csv.DictWriter(stream, fieldnames=list(row))
                if not exists:
                    writer.writeheader()
                writer.writerow(row)
            self.stop_event.wait(self.interval)


## 7. Идемпотентный SSH-туннель и PostgreSQL engine


In [ ]:
import atexit
import contextlib
import io
import socket as socket_module
import ssl

from sqlalchemy import bindparam, inspect, text
from sqlalchemy.engine import make_url
from sqlalchemy.ext.asyncio import AsyncEngine, create_async_engine

_SSH_TUNNEL: Any | None = globals().get("_SSH_TUNNEL")

def _free_local_port() -> int:
    with socket_module.socket() as sock:
        sock.bind(("localhost", 0))
        return int(sock.getsockname()[1])

def start_ssh_tunnel(config: dict[str, str | None]) -> Any:
    """Создать SSH tunnel на свободном loopback-порту, ключ держать в памяти."""
    import paramiko
    from sshtunnel import SSHTunnelForwarder
    key_text = config.get("private_key")
    key_file = config.get("private_key_file")
    pkey = None
    if key_text:
        for key_class in (paramiko.Ed25519Key, paramiko.RSAKey, paramiko.ECDSAKey):
            try:
                pkey = key_class.from_private_key(io.StringIO(str(key_text)))
                break
            except Exception:
                continue
        if pkey is None:
            raise RuntimeError("SSH private key не распознан.")
    tunnel = SSHTunnelForwarder(
        (str(config["host"]), int(config.get("port") or 22)),
        ssh_username=str(config["user"]), ssh_pkey=pkey,
        ssh_private_key=str(key_file) if key_file and not key_text else None,
        remote_bind_address=(str(config["db_host"]), int(config["db_port"] or 5432)),
        local_bind_address=("localhost", _free_local_port()),
    )
    tunnel.start()
    return tunnel

def stop_ssh_tunnel() -> None:
    """Идемпотентно закрыть текущий SSH tunnel."""
    global _SSH_TUNNEL
    if _SSH_TUNNEL is not None:
        with contextlib.suppress(Exception):
            _SSH_TUNNEL.stop()
        _SSH_TUNNEL = None

def get_or_reuse_ssh_tunnel(config: dict[str, str | None]) -> Any:
    """Переиспользовать активный tunnel или создать ровно один новый."""
    global _SSH_TUNNEL
    if _SSH_TUNNEL is not None and bool(getattr(_SSH_TUNNEL, "is_active", False)):
        return _SSH_TUNNEL
    stop_ssh_tunnel()
    _SSH_TUNNEL = start_ssh_tunnel(config)
    return _SSH_TUNNEL

atexit.register(stop_ssh_tunnel)

def build_database_url() -> str:
    """Собрать asyncpg URL из секретов, с прямым или SSH-подключением."""
    explicit = SECRETS.get("DATABASE_URL")
    if explicit:
        url = make_url(explicit)
        return str(url.set(drivername="postgresql+asyncpg"))
    required = ["DB_HOST", "DB_NAME", "DB_USER", "DB_PASSWORD"]
    missing = [name for name in required if not SECRETS.get(name)]
    if missing:
        raise RuntimeError("Не заданы параметры PostgreSQL: " + ", ".join(missing))
    host, port = str(SECRETS["DB_HOST"]), int(SECRETS.get("DB_PORT") or 5432)
    if str(SECRETS.get("SSH_ENABLED") or "").lower() in {"1", "true", "yes"}:
        tunnel = get_or_reuse_ssh_tunnel({
            "host": SECRETS.get("SSH_HOST"), "port": SECRETS.get("SSH_PORT") or "22",
            "user": SECRETS.get("SSH_USER"), "private_key": SECRETS.get("SSH_PRIVATE_KEY"),
            "private_key_file": SECRETS.get("SSH_PRIVATE_KEY_FILE"),
            "db_host": host, "db_port": str(port),
        })
        host, port = "localhost", int(tunnel.local_bind_port)
    return f"postgresql+asyncpg://{quote(str(SECRETS['DB_USER']))}:{quote(str(SECRETS['DB_PASSWORD']), safe='')}@{host}:{port}/{quote(str(SECRETS['DB_NAME']))}"

def create_postgres_engine(database_url: str) -> AsyncEngine:
    """Создать SQLAlchemy 2.x async engine с настроенным PostgreSQL SSL."""
    ssl_mode = str(SECRETS.get("DB_SSL_MODE") or "prefer")
    connect_args: dict[str, Any] = {"timeout": 30, "command_timeout": 120}
    if ssl_mode in {"require", "verify-ca", "verify-full"}:
        ssl_context = ssl.create_default_context(cafile=SECRETS.get("DB_SSL_CA_FILE") or None)
        if ssl_mode == "require":
            ssl_context.check_hostname = False
            ssl_context.verify_mode = ssl.CERT_NONE
        connect_args["ssl"] = ssl_context
    elif ssl_mode == "disable":
        connect_args["ssl"] = False
    return create_async_engine(database_url, pool_pre_ping=True, pool_size=2, max_overflow=0, connect_args=connect_args)

REQUIRED_COLUMNS = {
    "group_posts": {"id", "group_id", "community_vk_id", "published_at", "text"},
    "post_attachments": {"post_id", "position", "attachment_type"},
    "post_embeddings": {"post_id", "run_id", "model_name", "embedding_dim", "embedding_vector", "modality_profile"},
}

async def postgres_preflight(engine: AsyncEngine) -> dict[str, Any]:
    """Выполнить только read-only проверки схемы до загрузки модели."""
    started = time.perf_counter()
    async with engine.connect() as connection:
        await connection.execute(text("SET TRANSACTION READ ONLY"))
        await connection.execute(text("SELECT 1"))
        version = (await connection.execute(text("SELECT version()"))).scalar_one()
        def inspect_schema(sync_connection: Any) -> dict[str, Any]:
            inspector = inspect(sync_connection)
            tables = set(inspector.get_table_names())
            columns = {table: {item["name"] for item in inspector.get_columns(table)} if table in tables else set() for table in REQUIRED_COLUMNS}
            unique_constraints = inspector.get_unique_constraints("post_embeddings") if "post_embeddings" in tables else []
            indexes = inspector.get_indexes("post_embeddings") if "post_embeddings" in tables else []
            return {"tables": sorted(tables), "columns": {k: sorted(v) for k, v in columns.items()}, "unique_constraints": unique_constraints, "indexes": indexes}
        schema = await connection.run_sync(inspect_schema)
        existing_dimensions = []
        if "post_embeddings" in schema["tables"]:
            existing_dimensions = [int(row[0]) for row in (await connection.execute(text("SELECT DISTINCT embedding_dim FROM post_embeddings ORDER BY embedding_dim LIMIT 20"))).all()]
    for table, required in REQUIRED_COLUMNS.items():
        actual = set(schema["columns"].get(table, []))
        if not required <= actual:
            raise RuntimeError(f"Схема несовместима: {table}, отсутствуют {sorted(required - actual)}")
    unique_sets = [{*item.get("column_names", [])} for item in schema["unique_constraints"]]
    unique_sets += [{*item.get("column_names", [])} for item in schema["indexes"] if item.get("unique")]
    if {"post_id"} not in unique_sets:
        raise RuntimeError("Не подтверждена текущая уникальность post_embeddings(post_id).")
    incompatible_dimensions = [value for value in existing_dimensions if value != EMBEDDING_DIM]
    if incompatible_dimensions:
        raise RuntimeError(f"В post_embeddings найдены несовместимые embedding_dim: {incompatible_dimensions}")
    schema.update(postgres_version=str(version), embedding_dim=EMBEDDING_DIM, existing_embedding_dimensions=existing_dimensions, conflict_key=["post_id"], query_seconds=time.perf_counter() - started)
    return schema


## 8. Pydantic-контракты данных и manifest


In [ ]:
import hashlib
import uuid
from datetime import UTC, datetime, timedelta
from enum import StrEnum

import numpy as np
from pydantic import BaseModel, ConfigDict, Field

MANIFEST_VERSION = 1
RESUMABLE_STATUSES = {"running", "interrupted", "failed"}

class ModalityProfile(StrEnum):
    TEXT_ONLY = "text_only"
    TEXT_IMAGE = "text_image"
    TEXT_VIDEO = "text_video"
    TRIMODAL = "trimodal"
    IMAGE_ONLY = "image_only"
    VIDEO_ONLY = "video_only"
    EMPTY = "empty"

class PostAttachmentItem(BaseModel):
    model_config = ConfigDict(extra="ignore")
    position: int
    attachment_type: str
    vk_owner_id: int | None = None
    vk_attachment_id: int | None = None
    access_key: str | None = None
    duration: int | None = None
    width: int | None = None
    height: int | None = None
    title: str | None = None
    external_url: str | None = None
    attachment_metadata: dict[str, Any] = Field(default_factory=dict)

class MultimodalPost(BaseModel):
    model_config = ConfigDict(extra="ignore")
    post_id: int
    group_id: int
    community_vk_id: int
    subject: str
    published_at: datetime
    text: str = ""
    modality_profile: ModalityProfile
    attachments: list[PostAttachmentItem] = Field(default_factory=list)
    comments_count: int = 0
    likes_count: int = 0
    reposts_count: int = 0
    views_count: int = 0

class RunManifest(BaseModel):
    model_config = ConfigDict(extra="forbid")
    manifest_version: int = MANIFEST_VERSION
    run_id: str
    dataset_name: str
    model_name: str
    model_revision: str
    embedding_dim: int
    critical_config: dict[str, Any]
    critical_config_sha256: str
    dataset_sha256: str
    dataset_stats: dict[str, Any]
    schema_evidence: dict[str, Any] = Field(default_factory=dict)
    status: str = "initialized"
    created_at_utc: datetime = Field(default_factory=lambda: datetime.now(UTC))
    updated_at_utc: datetime = Field(default_factory=lambda: datetime.now(UTC))

class EncodedPair(BaseModel):
    model_config = ConfigDict(arbitrary_types_allowed=True)
    post: MultimodalPost
    embedding: Any

def canonical_sha256(payload: dict[str, Any]) -> str:
    """Хешировать JSON-совместимую конфигурацию канонически."""
    raw = json.dumps(payload, ensure_ascii=False, sort_keys=True, separators=(",", ":"), default=str).encode()
    return hashlib.sha256(raw).hexdigest()

def atomic_write_json(path: Path, payload: dict[str, Any]) -> None:
    """Атомарно записать JSON через временный файл, flush, fsync и os.replace."""
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + ".tmp")
    with temporary.open("w", encoding="utf-8") as stream:
        json.dump(payload, stream, ensure_ascii=False, indent=2, default=str)
        stream.flush()
        with contextlib.suppress(OSError):
            os.fsync(stream.fileno())
    os.replace(temporary, path)

def file_sha256(path: Path) -> str:
    """Потоково вычислить SHA-256 файла."""
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

CRITICAL_CONFIG = {
    "window_days": WINDOW_DAYS, "max_posts_per_group": MAX_POSTS_PER_GROUP,
    "embedding_dim": EMBEDDING_DIM, "gpu_batch_size": GPU_BATCH_SIZE,
    "gpu_min_batch_size": GPU_MIN_BATCH_SIZE, "db_batch_size": DB_BATCH_SIZE,
    "model_revision": MODEL_REVISION, "snapshot_sql_batch_size": SNAPSHOT_SQL_BATCH_SIZE,
}
CRITICAL_CONFIG_SHA256 = canonical_sha256(CRITICAL_CONFIG)


## 9. Формирование immutable dataset snapshot


In [ ]:
def determine_modality_profile(has_text: bool, has_photo: bool, has_video: bool) -> ModalityProfile:
    """Определить профиль модальности поста."""
    if has_text and has_photo and has_video: return ModalityProfile.TRIMODAL
    if has_text and has_photo: return ModalityProfile.TEXT_IMAGE
    if has_text and has_video: return ModalityProfile.TEXT_VIDEO
    if has_text: return ModalityProfile.TEXT_ONLY
    if has_photo and has_video: return ModalityProfile.TRIMODAL
    if has_photo: return ModalityProfile.IMAGE_ONLY
    if has_video: return ModalityProfile.VIDEO_ONLY
    return ModalityProfile.EMPTY

def generate_synthetic_posts(n: int = 32, seed: int = 42) -> list[MultimodalPost]:
    """Создать синтетические данные исключительно для smoke-режима."""
    rng = np.random.default_rng(seed)
    subjects = ["food_delivery", "food_service", "customer_acquisition", "tender_support"]
    now = utc_now()
    return [MultimodalPost(
        post_id=10_000 + index, group_id=index % 8 + 1, community_vk_id=index % 8 + 1,
        subject=subjects[index % len(subjects)], published_at=now - timedelta(hours=index),
        text=f"Синтетический smoke-пост {index}", modality_profile=ModalityProfile.TEXT_ONLY,
        likes_count=int(rng.integers(0, 100)),
    ) for index in range(n)]

POST_PAGE_SQL = text("""
    WITH ranked AS (
        SELECT p.id AS post_id, p.group_id, p.community_vk_id,
               COALESCE(gl.label, 'customer_acquisition') AS subject,
               p.published_at, p.text, p.comments_count, p.likes_count,
               p.reposts_count, p.views_count,
               ROW_NUMBER() OVER (PARTITION BY p.group_id ORDER BY p.published_at DESC, p.id DESC) AS rank_in_group
        FROM group_posts p
        JOIN group_candidates g ON g.id = p.group_id
        LEFT JOIN (SELECT group_id, MIN(label) AS label FROM group_labels GROUP BY group_id) gl ON gl.group_id = g.id
        WHERE g.classification_status = 'approved' AND p.published_at >= :cutoff
    )
    SELECT post_id, group_id, community_vk_id, subject, published_at, text,
           comments_count, likes_count, reposts_count, views_count
    FROM ranked WHERE rank_in_group <= :max_posts AND post_id > :last_post_id
    ORDER BY post_id ASC LIMIT :page_size
""")
ATTACHMENTS_SQL = text("""
    SELECT post_id, position, attachment_type, vk_owner_id, vk_attachment_id,
           access_key, duration, width, height, title, external_url, metadata AS attachment_metadata
    FROM post_attachments WHERE post_id IN :post_ids ORDER BY post_id, position
""").bindparams(bindparam("post_ids", expanding=True))

async def create_production_snapshot(engine: AsyncEngine, destination: Path) -> dict[str, Any]:
    """Потоково создать immutable JSONL snapshot keyset-страницами и bounded attachment batches."""
    if RUN_MODE != "production":
        raise RuntimeError("Production snapshot допустим только в RUN_MODE='production'.")
    temporary = destination.with_name(destination.name + ".tmp")
    cutoff = utc_now() - timedelta(days=WINDOW_DAYS)
    counts: dict[str, int] = defaultdict(int)
    groups: set[int] = set()
    digest = hashlib.sha256()
    post_count, last_post_id = 0, 0
    try:
        with temporary.open("wb") as stream:
            while True:
                async with engine.connect() as connection:
                    await connection.execute(text("SET TRANSACTION READ ONLY"))
                    rows = (await connection.execute(POST_PAGE_SQL, {
                        "cutoff": cutoff, "max_posts": MAX_POSTS_PER_GROUP,
                        "last_post_id": last_post_id, "page_size": SNAPSHOT_SQL_BATCH_SIZE,
                    })).mappings().all()
                    if not rows:
                        break
                    attachments_by_post: dict[int, list[PostAttachmentItem]] = defaultdict(list)
                    post_ids = [int(row["post_id"]) for row in rows]
                    for offset in range(0, len(post_ids), ATTACHMENT_POST_IDS_BATCH_SIZE):
                        chunk = post_ids[offset:offset + ATTACHMENT_POST_IDS_BATCH_SIZE]
                        attachment_rows = (await connection.execute(ATTACHMENTS_SQL, {"post_ids": chunk})).mappings().all()
                        for item in attachment_rows:
                            attachments_by_post[int(item["post_id"])].append(PostAttachmentItem.model_validate(dict(item)))
                for row in rows:
                    post_id = int(row["post_id"])
                    attachments = attachments_by_post.get(post_id, [])
                    text_value = str(row["text"] or "").strip()
                    profile = determine_modality_profile(bool(text_value), any(a.attachment_type == "photo" for a in attachments), any(a.attachment_type == "video" for a in attachments))
                    post = MultimodalPost(**dict(row), text=text_value, modality_profile=profile, attachments=attachments)
                    encoded = (post.model_dump_json() + "\n").encode("utf-8")
                    stream.write(encoded); digest.update(encoded)
                    post_count += 1; groups.add(post.group_id); counts[profile.value] += 1
                    last_post_id = post_id
            stream.flush()
            with contextlib.suppress(OSError): os.fsync(stream.fileno())
        os.replace(temporary, destination)
    except Exception:
        temporary.unlink(missing_ok=True)
        raise
    return {
        "dataset_name": DATASET_NAME, "sql_parameters": {"window_days": WINDOW_DAYS, "max_posts_per_group": MAX_POSTS_PER_GROUP},
        "window_start_utc": cutoff.isoformat(), "window_end_utc": utc_now().isoformat(),
        "post_count": post_count, "group_count": len(groups), "modality_distribution": dict(counts),
        "snapshot_created_at_utc": utc_now().isoformat(), "sha256": digest.hexdigest(),
    }

def create_smoke_snapshot(destination: Path) -> dict[str, Any]:
    """Атомарно создать небольшой synthetic snapshot без сети."""
    if RUN_MODE != "smoke":
        raise RuntimeError("Synthetic dataset запрещён вне smoke-режима.")
    temporary = destination.with_name(destination.name + ".tmp")
    posts = generate_synthetic_posts()
    with temporary.open("wb") as stream:
        for post in posts: stream.write((post.model_dump_json() + "\n").encode("utf-8"))
        stream.flush()
        with contextlib.suppress(OSError): os.fsync(stream.fileno())
    os.replace(temporary, destination)
    counts = defaultdict(int)
    for post in posts: counts[post.modality_profile.value] += 1
    return {"dataset_name": DATASET_NAME, "sql_parameters": {}, "window_start_utc": None, "window_end_utc": utc_now().isoformat(), "post_count": len(posts), "group_count": len({p.group_id for p in posts}), "modality_distribution": dict(counts), "snapshot_created_at_utc": utc_now().isoformat(), "sha256": file_sha256(destination)}


## 10. Поиск или создание run manifest


In [ ]:
def slugify(value: str) -> str:
    """Сформировать безопасный короткий slug для каталогов."""
    value = re.sub(r"[^a-zA-Z0-9._-]+", "-", value).strip("-.").lower()
    return value[:80] or "unnamed"

def load_manifest(path: Path) -> RunManifest:
    """Загрузить и валидировать manifest."""
    return RunManifest.model_validate_json(path.read_text(encoding="utf-8"))

def validate_manifest_compatibility(manifest: RunManifest) -> None:
    """Блокировать resume при несовпадении dataset/model/version/config."""
    if manifest.dataset_name != DATASET_NAME or manifest.model_name != MODEL_NAME:
        raise RuntimeError("Dataset/model manifest не совпадает с текущей конфигурацией.")
    if manifest.manifest_version != MANIFEST_VERSION:
        raise RuntimeError("Версия manifest несовместима.")
    if manifest.critical_config_sha256 != CRITICAL_CONFIG_SHA256:
        raise RuntimeError("Критичная конфигурация manifest несовместима.")

def find_resumable_manifest(storage_root: Path) -> tuple[Path, RunManifest] | None:
    """Найти единственный полностью совместимый активный запуск."""
    candidates: list[tuple[Path, RunManifest]] = []
    for path in storage_root.glob("*/*/*/run_manifest.json"):
        try:
            manifest = load_manifest(path)
        except Exception:
            continue
        if manifest.dataset_name == DATASET_NAME and manifest.model_name == MODEL_NAME and manifest.status in RESUMABLE_STATUSES:
            validate_manifest_compatibility(manifest)
            snapshot = path.parent / "dataset_snapshot.jsonl"
            if not snapshot.exists() or file_sha256(snapshot) != manifest.dataset_sha256:
                raise RuntimeError(f"Snapshot отсутствует или повреждён: {path.parent}")
            candidates.append((path, manifest))
    if len(candidates) > 1:
        raise RuntimeError("Найдено несколько совместимых активных запусков; укажите FORCE_NEW_RUN=True или завершите лишние вручную.")
    return candidates[0] if candidates else None

async def initialize_or_resume_run(engine: AsyncEngine | None, schema_evidence: dict[str, Any]) -> tuple[Path, RunManifest, bool]:
    """Восстановить run_id либо создать новый immutable snapshot и manifest."""
    matched = None if FORCE_NEW_RUN or not RESUME else find_resumable_manifest(STORAGE_ROOT)
    if matched:
        manifest_path, manifest = matched
        return manifest_path.parent, manifest, True
    run_id = f"vec-{utc_now():%Y%m%dT%H%M%SZ}-{uuid.uuid4().hex[:12]}"[:64]
    run_dir = STORAGE_ROOT / slugify(DATASET_NAME) / slugify(MODEL_NAME) / run_id
    run_dir.mkdir(parents=True, exist_ok=False)
    snapshot_path = run_dir / "dataset_snapshot.jsonl"
    if RUN_MODE == "smoke":
        stats = create_smoke_snapshot(snapshot_path)
    else:
        if engine is None:
            raise RuntimeError("Production snapshot требует доступного PostgreSQL; fallback на synthetic запрещён.")
        stats = await create_production_snapshot(engine, snapshot_path)
    dataset_hash = file_sha256(snapshot_path)
    (run_dir / "dataset_snapshot.sha256").write_text(dataset_hash + "  dataset_snapshot.jsonl\n", encoding="ascii")
    manifest = RunManifest(
        run_id=run_id, dataset_name=DATASET_NAME, model_name=MODEL_NAME,
        model_revision=MODEL_REVISION, embedding_dim=EMBEDDING_DIM,
        critical_config=CRITICAL_CONFIG, critical_config_sha256=CRITICAL_CONFIG_SHA256,
        dataset_sha256=dataset_hash, dataset_stats=stats, schema_evidence=schema_evidence,
        status="initialized",
    )
    atomic_write_json(run_dir / "run_manifest.json", manifest.model_dump(mode="json"))
    for filename in ("run.log", "events.jsonl", "failures.jsonl", "committed_batches.jsonl", "resource_metrics.csv"):
        (run_dir / filename).touch(exist_ok=True)
    return run_dir, manifest, False

def update_manifest(run_dir: Path, manifest: RunManifest, status: str) -> RunManifest:
    """Атомарно обновить только состояние текущего manifest."""
    updated = manifest.model_copy(update={"status": status, "updated_at_utc": utc_now()})
    atomic_write_json(run_dir / "run_manifest.json", updated.model_dump(mode="json"))
    return updated


## 11. DB-based resume


In [ ]:
COMPLETED_IDS_SQL = text("""
    SELECT post_id FROM post_embeddings
    WHERE run_id = :run_id AND model_name = :model_name
    ORDER BY post_id
""")

async def stream_completed_post_ids(engine: AsyncEngine, run_id: str, model_name: str) -> set[int]:
    """Получить подтверждённые DB post_id потоковым server-side cursor."""
    completed: set[int] = set()
    async with engine.connect() as connection:
        await connection.execute(text("SET TRANSACTION READ ONLY"))
        result = await connection.stream(COMPLETED_IDS_SQL, {"run_id": run_id, "model_name": model_name})
        async for partition in result.partitions(1000):
            completed.update(int(row.post_id) for row in partition)
    return completed

def iter_snapshot(path: Path, completed_ids: set[int] | None = None) -> Iterator[MultimodalPost]:
    """Потоково читать immutable snapshot, пропуская только DB-confirmed ids."""
    skipped = completed_ids or set()
    with path.open("r", encoding="utf-8") as stream:
        for line in stream:
            if line.strip():
                post = MultimodalPost.model_validate_json(line)
                if post.post_id not in skipped:
                    yield post


## 12. Мультимодальный dataset и обработка media


In [ ]:
from dataclasses import dataclass, field
from pathlib import Path
from collections.abc import Iterator, Sequence

from PIL import Image

@dataclass
class MultimodalBatchItem:
    post_id: int
    group_id: int
    subject: str
    text: str
    modality_profile: ModalityProfile
    images: list[np.ndarray] = field(default_factory=list)
    video_frames: list[np.ndarray] = field(default_factory=list)

class MediaResolver:
    """Разрешать только уже локализованные media; сетевых загрузок нет."""
    def __init__(self, cache_dir: Path) -> None:
        self.cache_dir = cache_dir
        self.cache_dir.mkdir(parents=True, exist_ok=True)

    def resolve(self, attachment: PostAttachmentItem) -> Path | None:
        suffix = ".jpg" if attachment.attachment_type == "photo" else ".mp4"
        key = f"{attachment.attachment_type}_{attachment.vk_owner_id}_{attachment.vk_attachment_id}_{attachment.position}{suffix}"
        candidate = self.cache_dir / key
        return candidate if candidate.is_file() and candidate.stat().st_size > 0 else None

def resize_frame(frame: np.ndarray, target_long_side: int = 448) -> np.ndarray:
    """Уменьшить RGB frame с сохранением пропорций."""
    height, width = frame.shape[:2]
    if max(height, width) <= target_long_side: return frame
    scale = target_long_side / max(height, width)
    image = Image.fromarray(frame).resize((max(1, round(width * scale)), max(1, round(height * scale))), Image.Resampling.BILINEAR)
    return np.asarray(image, dtype=np.uint8)

class MultimodalDataset:
    """Лениво подготовить один пост и доступные локальные изображения."""
    def __init__(self, resolver: MediaResolver) -> None: self.resolver = resolver

    def prepare(self, post: MultimodalPost) -> MultimodalBatchItem:
        images: list[np.ndarray] = []
        video_frames: list[np.ndarray] = []
        for attachment in post.attachments:
            path = self.resolver.resolve(attachment)
            if not path: continue
            if attachment.attachment_type == "photo":
                with Image.open(path) as image:
                    images.append(resize_frame(np.asarray(image.convert("RGB"))))
            # Видео ожидает заранее локализованные/извлечённые кадры; сетевых загрузок нет.
        return MultimodalBatchItem(post.post_id, post.group_id, post.subject, post.text, post.modality_profile, images, video_frames)


## 13. Энкодеры Qwen, Jina и Mock


In [ ]:
from abc import ABC, abstractmethod

def l2_normalize(matrix: np.ndarray, eps: float = 1e-12) -> np.ndarray:
    """L2-нормализовать строки матрицы."""
    norms = np.maximum(np.linalg.norm(matrix, axis=1, keepdims=True), eps)
    return np.asarray(matrix / norms, dtype=np.float32)

class BaseEncoder(ABC):
    def __init__(self, model_name: str, embedding_dim: int, device: str = "cpu", precision: str = "float32") -> None:
        self.model_name, self.embedding_dim, self.device, self.precision = model_name, embedding_dim, device, precision
        self.is_loaded = False
    @abstractmethod
    def load(self) -> None: ...
    @abstractmethod
    def encode_batch(self, items: Sequence[MultimodalBatchItem]) -> np.ndarray: ...

class MockEncoder(BaseEncoder):
    """Детерминированный CPU encoder для smoke/self-check."""
    def load(self) -> None: self.is_loaded = True
    def encode_batch(self, items: Sequence[MultimodalBatchItem]) -> np.ndarray:
        vectors = []
        for item in items:
            signature = f"{item.post_id}:{item.subject}:{len(item.images)}:{len(item.video_frames)}"
            seed = int(hashlib.sha256(signature.encode()).hexdigest(), 16) % (2**32)
            vectors.append(np.random.default_rng(seed).normal(size=self.embedding_dim))
        return l2_normalize(np.asarray(vectors, dtype=np.float32)) if vectors else np.empty((0, self.embedding_dim), np.float32)

class QwenEncoder(BaseEncoder):
    """Qwen VL adapter; веса загружаются только при явном production-запуске."""
    def load(self) -> None:
        import torch
        from transformers import AutoModel, AutoProcessor
        dtype = torch.bfloat16 if self.precision == "bfloat16" else torch.float16
        self.processor = AutoProcessor.from_pretrained(self.model_name, revision=MODEL_REVISION, trust_remote_code=True)
        self.model = AutoModel.from_pretrained(self.model_name, revision=MODEL_REVISION, torch_dtype=dtype, trust_remote_code=True, device_map=self.device)
        self.model.eval(); self.is_loaded = True
    def encode_batch(self, items: Sequence[MultimodalBatchItem]) -> np.ndarray:
        import torch
        vectors: list[np.ndarray] = []
        with torch.no_grad():
            for item in items:
                images = [Image.fromarray(value) for value in [*item.images, *item.video_frames]]
                kwargs = {"text": [item.text.strip() or "Компания"], "return_tensors": "pt", "padding": True}
                if images: kwargs["images"] = images
                inputs = self.processor(**kwargs).to(self.device)
                output = self.model(**inputs)
                hidden = output.last_hidden_state
                mask = inputs.get("attention_mask")
                pooled = hidden.mean(dim=1) if mask is None else (hidden * mask.unsqueeze(-1)).sum(dim=1) / mask.sum(dim=1, keepdim=True).clamp(min=1)
                vectors.append(pooled.squeeze(0).float().cpu().numpy())
        return l2_normalize(np.asarray(vectors))

class JinaEncoder(BaseEncoder):
    """Jina omni baseline adapter."""
    def load(self) -> None:
        from transformers import AutoModel
        self.model = AutoModel.from_pretrained(self.model_name, revision=MODEL_REVISION, trust_remote_code=True, device_map=self.device)
        self.model.eval(); self.is_loaded = True
    def encode_batch(self, items: Sequence[MultimodalBatchItem]) -> np.ndarray:
        raw = self.model.encode([item.text.strip() or "Компания" for item in items])
        matrix = np.asarray(raw, dtype=np.float32)
        if matrix.shape[1] != self.embedding_dim: raise ValueError("Размерность Jina не совпала с manifest.")
        return l2_normalize(matrix)


## 14. Отказоустойчивый GPU/DB runner


In [ ]:
import asyncio
import random
import signal
from sqlalchemy.dialects.postgresql import insert
from sqlalchemy.exc import DBAPIError, IntegrityError, OperationalError, ProgrammingError, StatementError

STOP_REQUESTED = False

def request_controlled_stop(signum: int | None = None, frame: Any | None = None) -> None:
    """Запросить остановку после уже начатого GPU batch."""
    global STOP_REQUESTED
    STOP_REQUESTED = True

for signal_name in ("SIGINT", "SIGTERM"):
    if hasattr(signal, signal_name):
        with contextlib.suppress(ValueError): signal.signal(getattr(signal, signal_name), request_controlled_stop)

class DatabaseWriter(ABC):
    @abstractmethod
    async def completed_ids(self, run_id: str, model_name: str) -> set[int]: ...
    @abstractmethod
    async def commit(self, pairs: Sequence[EncodedPair], run_id: str, model_name: str) -> int: ...

class FakeDatabaseWriter(DatabaseWriter):
    """Транзакционный fake writer для smoke/self-check без сервера."""
    def __init__(self, fail_next: bool = False) -> None:
        self.rows: dict[int, tuple[str, str]] = {}; self.commits = 0; self.fail_next = fail_next
    async def completed_ids(self, run_id: str, model_name: str) -> set[int]:
        return {post_id for post_id, owner in self.rows.items() if owner == (run_id, model_name)}
    async def commit(self, pairs: Sequence[EncodedPair], run_id: str, model_name: str) -> int:
        if self.fail_next: self.fail_next = False; raise OperationalError("fake transient", {}, RuntimeError("fake"))
        staged = dict(self.rows)
        for pair in pairs:
            owner = staged.get(pair.post.post_id)
            if owner and owner != (run_id, model_name): raise RuntimeError("post_id принадлежит другому run/model")
            staged[pair.post.post_id] = (run_id, model_name)
        self.rows = staged; self.commits += 1
        return len(pairs)

class PostgresDatabaseWriter(DatabaseWriter):
    """PostgreSQL writer с защитой владельца post_id и одной транзакцией на batch."""
    def __init__(self, engine: AsyncEngine, authorized: bool) -> None: self.engine, self.authorized = engine, authorized
    async def completed_ids(self, run_id: str, model_name: str) -> set[int]:
        return await stream_completed_post_ids(self.engine, run_id, model_name)
    async def commit(self, pairs: Sequence[EncodedPair], run_id: str, model_name: str) -> int:
        if not self.authorized: raise PermissionError("Mutating DB-запрос заблокирован confirmation gate.")
        post_ids = [pair.post.post_id for pair in pairs]
        ownership_sql = text("SELECT post_id, run_id, model_name FROM post_embeddings WHERE post_id IN :post_ids").bindparams(bindparam("post_ids", expanding=True))
        async with self.engine.connect() as connection:
            transaction = await connection.begin()
            try:
                owners = (await connection.execute(ownership_sql, {"post_ids": post_ids})).mappings().all()
                conflicts = [row for row in owners if row["run_id"] != run_id or row["model_name"] != model_name]
                if conflicts: raise RuntimeError("Обнаружен post_id другой модели/run; схема unique(post_id) не допускает безопасную запись.")
                table = table_clause("post_embeddings", column("post_id"), column("run_id"), column("model_name"), column("embedding_dim"), column("embedding_vector"), column("modality_profile"), column("updated_at"))
                values = [{"post_id": pair.post.post_id, "run_id": run_id, "model_name": model_name, "embedding_dim": len(pair.embedding), "embedding_vector": np.asarray(pair.embedding, dtype=np.float32).tolist(), "modality_profile": pair.post.modality_profile.value} for pair in pairs]
                statement = insert(table).values(values)
                statement = statement.on_conflict_do_update(index_elements=[table.c.post_id], set_={"embedding_dim": statement.excluded.embedding_dim, "embedding_vector": statement.excluded.embedding_vector, "modality_profile": statement.excluded.modality_profile, "updated_at": text("now()")}, where=(table.c.run_id == run_id) & (table.c.model_name == model_name))
                await connection.execute(statement)
                await transaction.commit()
            except Exception:
                await transaction.rollback(); raise
        return len(pairs)

from sqlalchemy import column, table as table_clause

def is_cuda_oom(error: BaseException) -> bool:
    """Распознать CUDA OOM без жёсткой зависимости от torch."""
    return "out of memory" in str(error).lower() and "cuda" in str(error).lower()

def clear_cuda_cache() -> None:
    """Освободить временный CUDA cache при наличии GPU."""
    with contextlib.suppress(Exception):
        import torch
        if torch.cuda.is_available(): torch.cuda.empty_cache()

def encode_with_fallback(encoder: BaseEncoder, dataset: MultimodalDataset, posts: Sequence[MultimodalPost], failures_path: Path, events_path: Path) -> list[EncodedPair]:
    """Вернуть точные пары post/embedding, уменьшая OOM batch 32→16→8→1."""
    def process(chunk: Sequence[MultimodalPost]) -> list[EncodedPair]:
        try:
            items = [dataset.prepare(post) for post in chunk]
            matrix = encoder.encode_batch(items)
            if len(matrix) != len(chunk): raise ValueError("Энкодер вернул неверное число векторов.")
            return [EncodedPair(post=post, embedding=embedding) for post, embedding in zip(chunk, matrix, strict=True)]
        except Exception as error:
            if is_cuda_oom(error) and len(chunk) > 1:
                append_jsonl(events_path, {"timestamp_utc": utc_now(), "event": "cuda_oom", "batch_size": len(chunk), "error_class": type(error).__name__})
                clear_cuda_cache()
                next_size = max(GPU_MIN_BATCH_SIZE, len(chunk) // 2) if len(chunk) > GPU_MIN_BATCH_SIZE else 1
                return [pair for offset in range(0, len(chunk), next_size) for pair in process(chunk[offset:offset + next_size])]
            if len(chunk) > 1:
                return [pair for post in chunk for pair in process([post])]
            post = chunk[0]
            append_jsonl(failures_path, {"timestamp_utc": utc_now(), "post_id": post.post_id, "group_id": post.group_id, "stage": "encode", "error_class": type(error).__name__, "error": REDACTOR.redact(str(error))[:500]})
            return []
    return process(posts)

def retryable_db_error(error: BaseException) -> bool:
    """Повторять только transient/timeout/deadlock; schema/data ошибки terminal."""
    if isinstance(error, (IntegrityError, ProgrammingError, StatementError, ValueError)): return False
    return isinstance(error, (OperationalError, DBAPIError, TimeoutError, ConnectionError))

async def commit_with_retry(writer: DatabaseWriter, pairs: Sequence[EncodedPair], run_id: str, model_name: str, events_path: Path, state: dict[str, Any]) -> int:
    """Commit с rollback внутри writer, exponential backoff и jitter."""
    for attempt in range(1, DB_MAX_RETRIES + 1):
        try:
            return await writer.commit(pairs, run_id, model_name)
        except Exception as error:
            append_jsonl(events_path, {"timestamp_utc": utc_now(), "event": "db_retry", "attempt": attempt, "error_class": type(error).__name__})
            if not retryable_db_error(error) or attempt == DB_MAX_RETRIES: raise
            state["retries"] += 1
            await asyncio.sleep(min(8.0, 0.5 * (2 ** (attempt - 1))) + random.random() * 0.25)
    raise RuntimeError("Недостижимое состояние retry.")

class ResumableRunner:
    """Независимые GPU/DB batching, bounded RAM, DB-confirmed progress."""
    def __init__(self, encoder: BaseEncoder, dataset: MultimodalDataset, writer: DatabaseWriter, run_dir: Path, run_id: str, model_name: str, state: dict[str, Any]) -> None:
        self.encoder, self.dataset, self.writer = encoder, dataset, writer
        self.run_dir, self.run_id, self.model_name, self.state = run_dir, run_id, model_name, state
        self.buffer: list[EncodedPair] = []; self.reservoir: list[np.ndarray] = []; self.seen_quality = 0

    async def _commit_prefix(self, size: int) -> int:
        pending = self.buffer[:size]
        started = time.perf_counter()
        committed = await commit_with_retry(self.writer, pending, self.run_id, self.model_name, self.run_dir / "events.jsonl", self.state)
        if committed != size: raise RuntimeError("DB writer подтвердил неверное число строк.")
        del self.buffer[:size]
        self.state["committed_count"] += committed; self.state["db_transactions"] += 1; self.state["db_buffer"] = len(self.buffer)
        append_jsonl(self.run_dir / "committed_batches.jsonl", {"timestamp_utc": utc_now(), "run_id": self.run_id, "count": committed, "post_id_min": min(p.post.post_id for p in pending), "post_id_max": max(p.post.post_id for p in pending), "duration_seconds": time.perf_counter() - started})
        return committed

    async def flush_full_batches(self) -> None:
        while len(self.buffer) >= DB_BATCH_SIZE: await self._commit_prefix(DB_BATCH_SIZE)

    async def finalize(self) -> None:
        if self.buffer: await self._commit_prefix(len(self.buffer))

    def _sample_quality(self, pairs: Sequence[EncodedPair]) -> None:
        for pair in pairs:
            self.seen_quality += 1
            vector = np.asarray(pair.embedding, dtype=np.float32)
            if len(self.reservoir) < QUALITY_RESERVOIR_SIZE: self.reservoir.append(vector)
            else:
                index = random.randrange(self.seen_quality)
                if index < QUALITY_RESERVOIR_SIZE: self.reservoir[index] = vector

    async def run(self, posts: Iterator[MultimodalPost]) -> None:
        batch: list[MultimodalPost] = []
        for post in posts:
            if STOP_REQUESTED: break
            batch.append(post)
            if len(batch) < GPU_BATCH_SIZE: continue
            await self._process_gpu_batch(batch); batch = []
        if batch: await self._process_gpu_batch(batch)
        await self.finalize()

    async def _process_gpu_batch(self, posts: Sequence[MultimodalPost]) -> None:
        started = time.perf_counter()
        pairs = encode_with_fallback(self.encoder, self.dataset, posts, self.run_dir / "failures.jsonl", self.run_dir / "events.jsonl")
        self.buffer.extend(pairs); self._sample_quality(pairs)
        self.state["processed_count"] += len(posts); self.state["failed_count"] += len(posts) - len(pairs)
        self.state["calculated_count"] += len(pairs); self.state["gpu_batches"] += 1; self.state["db_buffer"] = len(self.buffer)
        elapsed = time.perf_counter() - started
        self.state["throughput"] = len(posts) / max(elapsed, 1e-9)
        self.state["peak_throughput"] = max(float(self.state.get("peak_throughput", 0.0)), float(self.state["throughput"]))
        await self.flush_full_batches()


## 15. Запуск в smoke или production-режиме


In [ ]:
async def run_pipeline() -> dict[str, Any]:
    """Выполнить безопасный smoke или явно подтверждённый production run."""
    engine: AsyncEngine | None = None
    monitor: ResourceMonitor | None = None
    run_dir: Path | None = None
    manifest: RunManifest | None = None
    started = utc_now(); stage_durations: dict[str, float] = {}
    state: dict[str, Any] = {"processed_count": 0, "committed_count": 0, "failed_count": 0, "calculated_count": 0, "db_buffer": 0, "gpu_batch": GPU_BATCH_SIZE, "throughput": 0.0, "gpu_batches": 0, "db_transactions": 0, "retries": 0}
    try:
        schema_evidence: dict[str, Any] = {"mode": "smoke", "conflict_key": ["post_id"]}
        if RUN_MODE == "production":
            database_url = build_database_url()
            print("PostgreSQL:", mask_database_url(database_url))
            engine = create_postgres_engine(database_url)
            preflight_start = time.perf_counter(); schema_evidence = await postgres_preflight(engine); stage_durations["preflight"] = time.perf_counter() - preflight_start
        run_dir, manifest, resumed = await initialize_or_resume_run(engine, schema_evidence)
        logger = configure_logger(run_dir / "run.log")
        logger.info("run_id=%s stage=initialized gpu_batch=%s db_batch=%s resume=%s", manifest.run_id, GPU_BATCH_SIZE, DB_BATCH_SIZE, resumed)
        manifest = update_manifest(run_dir, manifest, "running")
        monitor = ResourceMonitor(run_dir / "resource_metrics.csv", STORAGE_ROOT, state, RESOURCE_SAMPLE_INTERVAL_SECONDS); monitor.start()
        writer: DatabaseWriter
        if RUN_MODE == "smoke": writer = FakeDatabaseWriter()
        else: writer = PostgresDatabaseWriter(engine, db_writes_authorized())  # type: ignore[arg-type]
        completed = await writer.completed_ids(manifest.run_id, manifest.model_name)
        state["existing_db_rows"] = len(completed); state["skipped_count"] = len(completed)
        cache_dir = (Path("/persistent_volume/vk-research-collector/cache/media") if ENVIRONMENT == "supercomputer" else run_dir / "media_cache")
        dataset = MultimodalDataset(MediaResolver(cache_dir))
        encoder: BaseEncoder = MockEncoder(MODEL_NAME, EMBEDDING_DIM) if RUN_MODE == "smoke" else QwenEncoder(MODEL_NAME, EMBEDDING_DIM, "cuda", "bfloat16")
        model_start = time.perf_counter(); encoder.load(); stage_durations["model_load"] = time.perf_counter() - model_start
        runner = ResumableRunner(encoder, dataset, writer, run_dir, manifest.run_id, manifest.model_name, state)
        inference_start = time.perf_counter()
        await runner.run(iter_snapshot(run_dir / "dataset_snapshot.jsonl", completed))
        stage_durations["inference_and_commit"] = time.perf_counter() - inference_start
        final_status = "interrupted" if STOP_REQUESTED else "completed"
        manifest = update_manifest(run_dir, manifest, final_status)
        quality = evaluate_quality(runner.reservoir, manifest)
        atomic_write_json(run_dir / "quality_report.json", quality)
        summary = build_summary(run_dir, manifest, state, started, stage_durations, final_status)
        atomic_write_json(run_dir / "summary.json", summary)
        return summary
    except (KeyboardInterrupt, asyncio.CancelledError):
        request_controlled_stop()
        if run_dir and manifest: update_manifest(run_dir, manifest, "interrupted")
        raise
    except Exception:
        if run_dir and manifest: update_manifest(run_dir, manifest, "failed")
        raise
    finally:
        if monitor: monitor.stop()
        if engine: await engine.dispose()
        stop_ssh_tunnel()

if EXECUTE_PIPELINE:
    PIPELINE_SUMMARY = await run_pipeline()
    print(json.dumps(PIPELINE_SUMMARY, ensure_ascii=False, indent=2))
else:
    print("Пайплайн не запущен. Проверьте конфигурацию, выполните self-checks и установите EXECUTE_PIPELINE=True.")


## 16. Диагностика качества


In [ ]:
def evaluate_quality(reservoir: Sequence[np.ndarray], manifest: RunManifest) -> dict[str, Any]:
    """Рассчитать bounded quality metrics без полной матрицы запуска."""
    if not reservoir:
        return {"run_id": manifest.run_id, "sample_size": 0, "status": "нет подтверждённых векторов"}
    matrix = np.asarray(reservoir, dtype=np.float32)
    norms = np.linalg.norm(matrix, axis=1)
    finite = np.isfinite(matrix)
    return {
        "run_id": manifest.run_id, "model_name": manifest.model_name,
        "sample_size": len(matrix), "embedding_dim": int(matrix.shape[1]),
        "nan_count": int(np.isnan(matrix).sum()), "inf_count": int(np.isinf(matrix).sum()),
        "zero_vector_count": int((norms < 1e-8).sum()),
        "is_l2_normalized": bool(np.all(np.abs(norms - 1.0) < 1e-3)),
        "mean_norm": float(norms.mean()), "finite_share": float(finite.mean()),
    }


## 17. Итоговый summary и корректное освобождение ресурсов


In [ ]:
def static_environment_report() -> dict[str, Any]:
    """Собрать статические версии и характеристики среды без секретов."""
    memory, swap = psutil.virtual_memory(), psutil.swap_memory()
    report: dict[str, Any] = {
        "environment": ENVIRONMENT, "os": platform.platform(), "python": sys.version,
        "architecture": platform.machine(), "hostname": socket.gethostname(),
        "git_sha": "unknown", "cpu_model": platform.processor(),
        "physical_cores": psutil.cpu_count(logical=False), "logical_cores": psutil.cpu_count(),
        "ram_total_bytes": memory.total, "swap_total_bytes": swap.total,
        "filesystems": [partition.mountpoint for partition in psutil.disk_partitions()],
        "model_revision": MODEL_REVISION,
    }
    with contextlib.suppress(Exception):
        report["git_sha"] = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True, timeout=3).strip()
    versions = {}
    for module_name in ("numpy", "pydantic", "sqlalchemy", "asyncpg", "psutil", "torch", "transformers"):
        with contextlib.suppress(Exception):
            module = __import__(module_name); versions[module_name] = getattr(module, "__version__", "unknown")
    report["library_versions"] = versions
    report["gpu"] = collect_gpu_metrics()
    with contextlib.suppress(Exception):
        import torch
        report.update(gpu_count=torch.cuda.device_count(), gpu_names=[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())], cuda_version=torch.version.cuda, cudnn_version=torch.backends.cudnn.version(), bf16_supported=torch.cuda.is_available() and torch.cuda.is_bf16_supported(), precision="bfloat16" if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else "float32")
    if shutil.which("nvidia-smi"):
        with contextlib.suppress(Exception):
            gpu_rows = subprocess.check_output(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader,nounits"], text=True, timeout=3).splitlines()
            report["nvidia_smi_gpus"] = gpu_rows
    return report

def build_summary(run_dir: Path, manifest: RunManifest, state: dict[str, Any], started: datetime, stage_durations: dict[str, float], final_status: str) -> dict[str, Any]:
    """Сформировать итоговый атомарно сохраняемый summary."""
    ended = utc_now(); duration = (ended - started).total_seconds()
    return {
        "run_id": manifest.run_id, "dataset": manifest.dataset_name, "model": manifest.model_name,
        "start_utc": started.isoformat(), "end_utc": ended.isoformat(), "total_duration_seconds": duration,
        "stage_durations_seconds": stage_durations, "snapshot_posts": manifest.dataset_stats.get("post_count", 0),
        "existing_db_rows": state.get("existing_db_rows", 0), "calculated": state.get("calculated_count", 0),
        "committed": state.get("committed_count", 0), "skipped": state.get("skipped_count", 0),
        "failures": state.get("failed_count", 0), "gpu_batches": state.get("gpu_batches", 0),
        "db_transactions": state.get("db_transactions", 0), "retries": state.get("retries", 0),
        "average_throughput": state.get("calculated_count", 0) / max(duration, 1e-9),
        "peak_throughput": state.get("peak_throughput", 0.0), "peak_ram_bytes": state.get("peak_ram_bytes", psutil.Process().memory_info().rss),
        "peak_vram_bytes": state.get("peak_vram_bytes", collect_gpu_metrics().get("vram_max_allocated_bytes")),
        "effective_gpu_batch": state.get("gpu_batch", GPU_BATCH_SIZE), "db_batch": DB_BATCH_SIZE,
        "max_recompute_window": DB_BATCH_SIZE - 1, "final_status": final_status,
        "artifacts": {name: str(run_dir / name) for name in ("run_manifest.json", "run.log", "events.jsonl", "resource_metrics.csv", "summary.json", "failures.jsonl", "dataset_snapshot.jsonl", "dataset_snapshot.sha256", "committed_batches.jsonl", "quality_report.json")},
        "environment": static_environment_report(),
    }

print("При завершении summary и manifest записываются атомарно; monitor, engine и SSH tunnel закрываются в finally.")


## 18. Встроенные self-checks и инструкция по resume


In [ ]:
async def run_self_checks() -> None:
    """Проверить resume/buffer/fallback без сети, сервера и реального GPU."""
    assert GPU_BATCH_SIZE == 32
    assert DB_BATCH_SIZE == 500
    with tempfile.TemporaryDirectory() as temporary_directory:
        root = Path(temporary_directory)
        run_dir = root / "dataset" / "model" / "self-check-run"
        run_dir.mkdir(parents=True)
        for filename in ("events.jsonl", "failures.jsonl", "committed_batches.jsonl"): (run_dir / filename).touch()
        writer = FakeDatabaseWriter()
        encoder = MockEncoder("model", 16); encoder.load()
        dataset = MultimodalDataset(MediaResolver(run_dir / "media"))
        state = {"processed_count": 0, "committed_count": 0, "failed_count": 0, "calculated_count": 0, "db_buffer": 0, "gpu_batch": 32, "throughput": 0.0, "gpu_batches": 0, "db_transactions": 0, "retries": 0}
        runner = ResumableRunner(encoder, dataset, writer, run_dir, "self-check-run", "model", state)
        posts = generate_synthetic_posts(512)
        pairs = [EncodedPair(post=post, embedding=np.ones(16, np.float32) / 4) for post in posts]
        runner.buffer = pairs[:499]; await runner.flush_full_batches(); assert writer.commits == 0 and len(runner.buffer) == 499
        runner.buffer.append(pairs[499]); await runner.flush_full_batches(); assert writer.commits == 1 and len(runner.buffer) == 0
        runner.buffer = list(pairs); await runner.flush_full_batches(); assert writer.commits == 2 and len(runner.buffer) == 12
        await runner.finalize(); assert writer.commits == 3 and len(runner.buffer) == 0
        class AlwaysFailWriter(FakeDatabaseWriter):
            async def commit(self, pairs: Sequence[EncodedPair], run_id: str, model_name: str) -> int:
                raise ValueError("fake invalid data")
        failing = AlwaysFailWriter(); failed_runner = ResumableRunner(encoder, dataset, failing, run_dir, "x", "model", state)
        failed_runner.buffer = pairs[:10]
        old_retries = globals()["DB_MAX_RETRIES"]; globals()["DB_MAX_RETRIES"] = 1
        journal_before = (run_dir / "committed_batches.jsonl").read_text(encoding="utf-8")
        try:
            try: await failed_runner.finalize()
            except ValueError: pass
            assert len(failed_runner.buffer) == 10
            assert (run_dir / "committed_batches.jsonl").read_text(encoding="utf-8") == journal_before
        finally: globals()["DB_MAX_RETRIES"] = old_retries
        writer.rows[posts[0].post_id] = ("resume", "model")
        assert posts[0].post_id in await writer.completed_ids("resume", "model")
        snapshot = run_dir / "dataset_snapshot.jsonl"; create_smoke_snapshot(snapshot)
        manifest = RunManifest(run_id="self-check-run", dataset_name=DATASET_NAME, model_name=MODEL_NAME, model_revision=MODEL_REVISION, embedding_dim=EMBEDDING_DIM, critical_config=CRITICAL_CONFIG, critical_config_sha256=CRITICAL_CONFIG_SHA256, dataset_sha256=file_sha256(snapshot), dataset_stats={})
        atomic_write_json(run_dir / "run_manifest.json", manifest.model_dump(mode="json")); restored = load_manifest(run_dir / "run_manifest.json"); assert restored.run_id == manifest.run_id
        mismatch = manifest.model_copy(update={"model_name": "other"}); atomic_write_json(run_dir / "mismatch.json", mismatch.model_dump(mode="json"))
        try: validate_manifest_compatibility(load_manifest(run_dir / "mismatch.json"))
        except RuntimeError: pass
        else: raise AssertionError("Dataset/model mismatch не заблокирован.")
        assert not (run_dir / "run_manifest.json.tmp").exists()
        original_environment = globals()["ENVIRONMENT"]
        try:
            globals()["ENVIRONMENT"] = "supercomputer"
            assert Path("/persistent_volume/vk-research-collector/vectorization_runs") == Path("/persistent_volume") / "vk-research-collector/vectorization_runs"
        finally: globals()["ENVIRONMENT"] = original_environment
        secret_marker = uuid.uuid4().hex; local_redactor = SecretRedactor([secret_marker]); assert secret_marker not in local_redactor.redact(f"password={secret_marker}")
        class PartialEncoder(MockEncoder):
            def encode_batch(self, items: Sequence[MultimodalBatchItem]) -> np.ndarray:
                if len(items) > 1: raise RuntimeError("batch failure")
                if items[0].post_id == posts[1].post_id: raise RuntimeError("item failure")
                return super().encode_batch(items)
        partial = PartialEncoder("partial", 16); partial.load()
        successful = encode_with_fallback(partial, dataset, posts[:3], run_dir / "failures.jsonl", run_dir / "events.jsonl")
        assert [pair.post.post_id for pair in successful] == [posts[0].post_id, posts[2].post_id]
        assert all(len(pair.embedding) == 16 for pair in successful)
    print("Self-checks пройдены: 15/15. Сеть, серверная БД и реальный GPU не использовались.")

await run_self_checks()


### Как возобновить запуск

1. Не меняйте `DATASET_NAME`, `MODEL_NAME`, `MODEL_REVISION` и critical config.
2. Оставьте `RESUME=True` и `FORCE_NEW_RUN=False`.
3. Повторно выполните ячейки. Будет выбран единственный совместимый manifest,
   проверены snapshot и SHA-256, затем из PostgreSQL потоково прочитаны подтверждённые `post_id`.
4. Если совместимых активных manifests несколько, notebook остановится и потребует
   явного решения. Для отдельного нового snapshot установите `FORCE_NEW_RUN=True`.
5. `committed_batches.jsonl` — аудит после COMMIT, но не источник истины.

Финальная ячейка `run_pipeline()` выводит компактный русский summary и абсолютные пути
ко всем артефактам. При production-сбое соединения synthetic fallback не выполняется.


In [ ]:
final_summary = globals().get("PIPELINE_SUMMARY")
if final_summary:
    print(f"Итог запуска {final_summary['run_id']}: {final_summary['final_status']}")
    print(f"Snapshot: {final_summary['snapshot_posts']}; committed: {final_summary['committed']}; failures: {final_summary['failures']}")
    print("Артефакты:")
    for artifact_name, artifact_path in final_summary["artifacts"].items():
        print(f"  {artifact_name}: {artifact_path}")
else:
    print("Безопасный smoke/self-check завершён. Основной pipeline намеренно не запускался; артефакты появятся после EXECUTE_PIPELINE=True.")
